# 02 — 모듈 ① KcELECTRA + Focal Loss 학습 (Colab A100)

**실행 환경**: 둘 다 가능
1. **VS Code + Google Colab 확장** (`Google.colab`) — 로컬 노트북, 원격 Colab 런타임
2. **브라우저 Colab** (`colab.research.google.com`) — 동일 노트북 업로드

**런타임 요건**: GPU (Pro: A100 권장 / Free: T4 가능, 학습 시간 약 3~5배 늘어남)

**산출물**:
- 학습 체크포인트 (Colab 디스크 + Drive 백업)
- 평가 메트릭 JSON
- Confusion matrix figure

**선행 문서**: [docs/label_mapping.md](../docs/label_mapping.md), [docs/emergency_scenarios.md](../docs/emergency_scenarios.md), [reports/d1_report.md](../reports/d1_report.md)

## 0. 환경 검증 — GPU 확인

In [2]:
!nvidia-smi | head -15
import torch
import sys
sys.path.insert(0, '~/thisabled-ai')


print(f"\nPyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sun May 31 20:17:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             48W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Repo Clone (private)

GitHub Personal Access Token 필요 (private repo). 토큰은 반복 노출 방지를 위해 `getpass`로 입력받음.

**재실행 안전**: 이미 clone되어 있으면 skip.

In [2]:
import os, getpass, subprocess
from pathlib import Path

REPO_OWNER = "threeGuineas"  # ← 본인 GitHub username으로 수정
REPO_NAME = "thisabled-ai"
REPO_DIR = Path(f"/content/{REPO_NAME}")

if REPO_DIR.exists():
    print(f"Repo already cloned at {REPO_DIR} — pulling latest")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
else:
    token = os.environ.get("GITHUB_TOKEN") or getpass.getpass("GitHub Personal Access Token: ")
    url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.run(["git", "clone", url, str(REPO_DIR)], check=True)
    print(f"Cloned to {REPO_DIR}")

%cd {REPO_DIR}
!git log --oneline -3

Repo already cloned at /content/thisabled-ai — pulling latest
/content/thisabled-ai
6000fc1 (HEAD -> main, origin/main, origin/HEAD) D1: 프로젝트 스캐폴드 + 시드 데이터 파이프라인 + KcELECTRA 학습 골격
defdc89 Initial commit


## 2. 의존성 설치

Colab은 자체 Python 환경 → 매번 install. `requirements-colab.txt`가 핵심 패키지만 포함하므로 빠름.

In [3]:
!pip install -q -r requirements-colab.txt
# transformers Trainer가 요구하지만 requirements-colab.txt에 없는 dep 보강
!pip install -q "accelerate>=0.26"
print("✓ 의존성 설치 완료")

✓ 의존성 설치 완료


## 3. 시드 데이터 다운로드 + processed parquet 빌드

둘 다 재실행 안전 (sha256 검증, 이미 있으면 skip).

In [4]:
!python scripts/download_seed_datasets.py
!python scripts/build_processed_dataset.py

target: /content/thisabled-ai/data/raw
[unsmile/train] skip (exists, 1,848,905 bytes)
[unsmile/valid] skip (exists, 455,315 bytes)
[kold] skip (exists, sha256 verified)
done.
=== Splits saved ===
[train] n=47,336  labels={0: 19836, 1: 9460, 2: 18040}  sources={'kold': 32291, 'unsmile': 15045}  -> data/processed/train.parquet
[val] n=5,917  labels={0: 2479, 1: 1183, 2: 2255}  sources={'kold': 4111, 'unsmile': 1806}  -> data/processed/val.parquet
[test] n=5,918  labels={0: 2480, 1: 1183, 2: 2255}  sources={'kold': 4027, 'unsmile': 1891}  -> data/processed/test.parquet


## 4. 스모크 테스트 — 모델·토크나이저·데이터 한 번 통과

본 학습 전 1분 안에 파이프라인 검증. 실패하면 본 학습 시작 안 함.

In [5]:
import sys, yaml, torch, pandas as pd
sys.path.insert(0, ".")
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from src.models.focal_loss import FocalLoss

with open("configs/module1_kcelectra.yaml") as f:
    cfg = yaml.safe_load(f)

model_name = cfg["model"]["name"]
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=cfg["model"]["num_labels"]
).cuda()

df = pd.read_parquet("data/processed/train.parquet").head(8)
enc = tokenizer(
    df["text"].tolist(),
    padding=True, truncation=True, max_length=cfg["model"]["max_length"],
    return_tensors="pt",
).to("cuda")
labels = torch.tensor(df["label"].values, dtype=torch.long, device="cuda")

out = model(**enc)
focal = FocalLoss(gamma=cfg["loss"]["focal_gamma"])
loss = focal(out.logits, labels)
loss.backward()

print(f"Logits shape: {tuple(out.logits.shape)}")
print(f"Focal loss: {loss.item():.4f}")
print("✓ 스모크 테스트 통과 — 본 학습 진행 가능")

del model, out, loss
torch.cuda.empty_cache()

Loading beomi/KcELECTRA-base-v2022...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Logits shape: (8, 4)
Focal loss: 0.7674
✓ 스모크 테스트 통과 — 본 학습 진행 가능


## 5. (선택) Drive 마운트 — 체크포인트 백업용

Colab 디스크는 세션 종료 시 휘발. 체크포인트는 Drive로 백업 권장.  
**선택 사항**: 안 마운트해도 학습은 진행됨, 백업만 skip.

In [6]:
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    from pathlib import Path
    Path("/content/drive/MyDrive/ThisAbled/checkpoints").mkdir(parents=True, exist_ok=True)
    print("✓ Drive 마운트 완료")
except (ImportError, Exception) as e:
    print(f"⚠ Drive 마운트 skip ({e}) — 체크포인트 백업은 수동")

Mounted at /content/drive
✓ Drive 마운트 완료


## 6. 본 학습 — KcELECTRA + Focal Loss (5 epoch)

config: [configs/module1_kcelectra.yaml](../configs/module1_kcelectra.yaml)  
예상 시간: A100 ≈ 15~25분, T4 ≈ 60~90분 (5 epoch × 47K 샘플)

In [7]:
from src.training.trainer import train_module1

result = train_module1("configs/module1_kcelectra.yaml", project_root=".")

import json
print("\n=== 학습 완료 ===")
print(json.dumps(result["eval_metrics"], indent=2, default=str, ensure_ascii=False))

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at beomi/KcELECTRA-base-v2022 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/content/thisabled-ai/src/training/trainer.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalLossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Macro F1,Emergency Recall,Auc Pr,F1 Class0,F1 Class1,F1 Class2
1,0.237400,0.233477,0.742205,0.000000,0.834145,0.793679,0.638878,0.794057
2,0.161200,0.230307,0.756861,0.000000,0.841367,0.820280,0.649958,0.800346
3,0.118500,0.296696,0.747790,0.000000,0.821022,0.821608,0.624324,0.797437
4,0.078400,0.348052,0.749239,0.000000,0.817000,0.827011,0.629437,0.791268
5,0.051800,0.426175,0.745377,0.000000,0.805547,0.823362,0.622336,0.790432



=== 학습 완료 ===
{
  "eval_loss": 0.23030680418014526,
  "eval_macro_f1": 0.7568611791984217,
  "eval_emergency_recall": 0.0,
  "eval_auc_pr": 0.8413673025434473,
  "eval_f1_class0": 0.8202802967848309,
  "eval_f1_class1": 0.6499575191163977,
  "eval_f1_class2": 0.8003457216940363,
  "eval_runtime": 3.39,
  "eval_samples_per_second": 1745.417,
  "eval_steps_per_second": 27.433,
  "epoch": 5.0
}


## 7. Test set 평가 + Confusion Matrix

val로 best 모델을 골랐으니, hold-out test로 최종 메트릭.

In [9]:
import json
from pathlib import Path

ckpt_root = Path(result["checkpoint_dir"])
state_file = ckpt_root / "trainer_state.json"

if state_file.exists():
    state = json.loads(state_file.read_text())
    best = state.get("best_model_checkpoint")
    if best and Path(best).exists():
        ckpt = Path(best)
    else:
        # best 경로 무효면 최신 checkpoint-* 사용
        ckpt = sorted(ckpt_root.glob("checkpoint-*"),
                      key=lambda p: int(p.name.split("-")[1]))[-1]
else:
    ckpt = sorted(ckpt_root.glob("checkpoint-*"),
                  key=lambda p: int(p.name.split("-")[1]))[-1]

print(f"Best checkpoint: {ckpt}")
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt).cuda().eval()

Best checkpoint: models/checkpoints/module1/checkpoint-7400


In [4]:
import json
import torch, numpy as np
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding
from src.training.dataset import RiskTextDataset
from src.evaluation.metrics import compute_classification_metrics

# Use the `ckpt` selected in the previous cell (do not override with result["checkpoint_dir"]).
# `ckpt` is expected to be a Path pointing to the chosen checkpoint folder (e.g., checkpoint-*).
tokenizer = AutoTokenizer.from_pretrained(ckpt)
model = AutoModelForSequenceClassification.from_pretrained(ckpt).cuda().eval()

test_ds = RiskTextDataset("data/processed/test.parquet", tokenizer, cfg["model"]["max_length"]) 
loader = DataLoader(test_ds, batch_size=cfg["training"]["eval_batch_size"],
                    collate_fn=DataCollatorWithPadding(tokenizer))

all_logits, all_labels = [], []
with torch.no_grad():
    for batch in loader:
        labels_b = batch.pop("labels")
        batch = {k: v.cuda() for k, v in batch.items()}
        out = model(**batch)
        all_logits.append(out.logits.cpu().numpy())
        all_labels.append(labels_b.numpy())

logits = np.concatenate(all_logits)
y_true = np.concatenate(all_labels)
proba = torch.softmax(torch.tensor(logits), dim=-1).numpy()
y_pred = logits.argmax(axis=-1)

metrics = compute_classification_metrics(y_true, y_pred, proba)
print(json.dumps(metrics, indent=2, ensure_ascii=False))

# 데이터셋별 분리 보고 (label_mapping.md §6.L4 권고)
import pandas as pd
test_df = pd.read_parquet("data/processed/test.parquet").reset_index(drop=True)
for src in ("unsmile", "kold"):
    mask = (test_df["source"] == src).values
    m = compute_classification_metrics(y_true[mask], y_pred[mask], proba[mask])
    print(f"\n[{src}] macro_f1={m['macro_f1']:.4f}  emergency_recall={m['emergency_recall']:.4f}")

ModuleNotFoundError: No module named 'src'

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from pathlib import Path

try:
    import koreanize_matplotlib  # noqa
except ImportError:
    plt.rcParams["font.family"] = "DejaVu Sans"

cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
names = ["정상", "주의", "경고", "긴급"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=names, yticklabels=names, ax=ax, cbar=False)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Module ① Baseline — Macro-F1 {metrics['macro_f1']:.3f}")
plt.tight_layout()

out_dir = Path("reports/figures")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "module1_confusion_baseline.png", dpi=120)
plt.show()

## 8. 메트릭 JSON 저장 + 체크포인트 Drive 백업

In [ ]:
import datetime, shutil, json, os
from pathlib import Path

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# 메트릭 저장 (repo 안 — git commit 가능)
metrics_dir = Path("reports/validation_reports/module1")
metrics_dir.mkdir(parents=True, exist_ok=True)
metrics_path = metrics_dir / f"baseline_{ts}.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump({
        "timestamp": ts,
        "model": cfg["model"]["name"],
        "checkpoint": str(result["checkpoint_dir"]),
        "test_metrics": metrics,
        "train_metrics": {k: float(v) if hasattr(v, "item") else v
                          for k, v in result["train_metrics"].items()},
    }, f, indent=2, ensure_ascii=False, default=str)
print(f"메트릭 저장: {metrics_path}")

# Drive 백업 (마운트되어 있을 때만)
if os.path.exists("/content/drive/MyDrive"):
    drive_root = Path("/content/drive/MyDrive/ThisAbled/checkpoints")
    drive_root.mkdir(parents=True, exist_ok=True)
    dest = drive_root / f"module1_baseline_{ts}"
    shutil.copytree(result["checkpoint_dir"], dest, dirs_exist_ok=True)
    print(f"체크포인트 Drive 백업: {dest}")
    # 메트릭도 Drive에
    shutil.copy(metrics_path, dest / metrics_path.name)
else:
    print("⚠ Drive 미마운트 — 체크포인트 백업 skip (Colab 세션 종료 시 휘발)")

## 9. 다음 단계 (D2 잔여 / D3)

- [ ] **D2-잔여**: Focal Loss vs CE / class weight 비교 실험 (config 변경하고 본 노트북 재실행 → 메트릭 비교)
- [ ] **D2-잔여**: `reports/validation_reports/module1/baseline.md` 작성 (위 JSON + 데이터셋별 표 + figure 첨부)
- [ ] **D2-여유**: 소규모 GPT-4o 합성 (긴급 시나리오 정의서 §5 기반)
- [ ] **D3**: LightGBM Stacking, 모듈 ① 최종 학습, 모듈 ② 시작

In [ ]:
import getpass
import os

# Google AI Studio 무료 발급: https://aistudio.google.com/apikey
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")

!pip install -q google-genai
print("✓ GEMINI_API_KEY 설정 + google-genai 설치 완료")

In [ ]:
# 호출 수·예상 시간만 확인 (API 호출 없음).
# 헤더에 'gemini-2.0-flash (무료 tier, 15 RPM)' 가 떠야 최신 코드. gpt-4o-mini면 git pull 필요.
!python scripts/synthesize_emergency.py --dry-run

In [ ]:
# 실제 합성 (~5분). 중간 실패해도 누적 저장 → 재실행 시 이어서 생성.
# 빈 배치가 연속 5회면 해당 split 중단(quota 보호) → 로그에 '✗ 연속 5회' 뜨면 재실행.
!python scripts/synthesize_emergency.py

In [ ]:
import json
from pathlib import Path

OUT = Path("data/synthetic/emergency")
print("=== 카테고리/split 카운트 ===")
total = 0
for cat in ("3a", "3b", "3c", "3d", "boundary"):
    for split in ("train", "val", "test"):
        p = OUT / cat / f"{split}.jsonl"
        n = sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0
        total += n
        print(f"  {cat}/{split}: {n}")
print(f"총 {total}건 (목표 ~1,170)")

print("\n=== 카테고리별 train 샘플 3건 ===")
for cat in ("3a", "3b", "3c", "3d", "boundary"):
    p = OUT / cat / "train.jsonl"
    if not p.exists():
        continue
    rows = [json.loads(ln) for ln in p.open(encoding="utf-8")][:3]
    print(f"\n[{cat}]")
    for r in rows:
        extra = f" (label={r['label']})" if cat == "boundary" else ""
        print(f"  · {r['text']}{extra}")

In [ ]:
import os
import shutil
from pathlib import Path

src = Path("data/synthetic/emergency")
if os.path.exists("/content/drive/MyDrive"):
    dest = Path("/content/drive/MyDrive/ThisAbled/synthetic/emergency")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dest, dirs_exist_ok=True)
    print(f"✓ Drive 백업 완료: {dest}")
else:
    print("⚠ Drive 미마운트 (§5 셀 실행) — 합성 데이터 백업 skip")